# Создание синтетического датасета

Для демонстрации возможностей стратификации и метода CUPED при снижении дисперсии метрики в A/B-тестах был разработан синтетический датасет на основе данных B2C e-commerce сервиса (интернет-магазина книг или автозапчастей).

Исходные условия генерации:
- в качестве целевой метрики выступает средний чек пользователя;
- длительность эксперимента и период сбора исторических данных равны 1 месяцу;
- в сервисе действует ступенчатая система скидок в зависимости от накопленной суммы покупок: 0% — до 5 000 руб.; 3% — от 5 000 до 10 000 руб.; 5% — от 10 000 до 15 000 руб.; 7% — свыше 15 000 руб.

Ниже приведен полный код генерации исторических данных (pre-period) и данных экспериментального периода (test-period) с заложенным эффектом +0,5% в тестовой группе.

In [3]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)

HISTORY_START = datetime(2026, 6, 1, 0, 0, 0)
HISTORY_END = datetime(2026, 6, 30, 23, 59, 59)
EXPERIMENT_START = datetime(2025, 7, 1, 0, 0, 0)
EXPERIMENT_END = datetime(2025, 7, 31, 23, 59, 59)

MIN_ORDER, MAX_ORDER, CV = 100, 20000, 0.55
DISCOUNT_THRESHOLDS = [0, 5000, 10000, 15000]
DISCOUNT_VALUES = [0, 3, 5, 7]
RELATIVE_EFFECT = 0.005
N_USERS = 6000
VISITS_PER_USER_RANGE = (1, 8)

def random_datetimes_simple(start_date, end_date, n):
    """
    Генерирует случайные datetime 
    """
    total_seconds = int((end_date - start_date).total_seconds())
    random_seconds = np.random.randint(0, total_seconds, n).tolist()
    datetimes = [start_date + timedelta(seconds=seconds) for seconds in random_seconds]
    return sorted(datetimes)

def generate_order_amount():
    mean_log = np.log(3000)
    sigma_log = np.sqrt(np.log(1 + CV**2))
    amount = np.exp(np.random.normal(mean_log, sigma_log))
    return np.clip(amount, MIN_ORDER, MAX_ORDER)

def get_discount_by_total_spent(total_spent):
    for i, threshold in enumerate(DISCOUNT_THRESHOLDS[::-1]):
        if total_spent >= threshold:
            return DISCOUNT_VALUES[::-1][i]
    return 0

def generate_user_history(user_id, start_date, end_date):
    n_visits = np.random.randint(VISITS_PER_USER_RANGE[0], VISITS_PER_USER_RANGE[1] + 1)
    visit_dates = random_datetimes_simple(start_date, end_date, n_visits)
    
    records = []
    total_spent = 0
    for visit_time in visit_dates:
        amount = generate_order_amount()
        total_spent += amount
        records.append({
            'datetime': visit_time,
            'user_id': user_id,
            'order_amount': round(amount, 2),
            'period': 'history'
        })
    return records, total_spent

def generate_experiment_visits(user_id, discount_percent, group, start_date, end_date):
    n_visits = np.random.randint(VISITS_PER_USER_RANGE[0], VISITS_PER_USER_RANGE[1] + 1)
    visit_dates = random_datetimes_simple(start_date, end_date, n_visits)
    
    records = []
    for visit_time in visit_dates:
        base_amount = generate_order_amount()
        final_amount = base_amount * (1 - discount_percent / 100.0)
        
        if group == 'test':
            actual_effect = np.random.normal(RELATIVE_EFFECT, RELATIVE_EFFECT * 0.2)
            final_amount *= (1 + actual_effect)
        
        records.append({
            'datetime': visit_time,
            'user_id': user_id,
            'order_amount': round(final_amount, 2),
            'period': 'experiment',
            'strata': f"{discount_percent}%",
            'group': group
        })
    return records

# генерируем исторические данные
user_discounts = []
all_history_records = []
all_experiment_records = []

for user_id in range(1, N_USERS + 1):
    history_visits, total_spent = generate_user_history(f"user_{user_id}", HISTORY_START, HISTORY_END)
    all_history_records.extend(history_visits)
    discount = get_discount_by_total_spent(total_spent)
    user_discounts.append((f"user_{user_id}", discount))

# стратификация
discount_df = pd.DataFrame(user_discounts, columns=['user_id', 'discount_percent'])
discount_df['group'] = 'control'

for discount_val in discount_df['discount_percent'].unique():
    mask = discount_df['discount_percent'] == discount_val
    users_idx = discount_df[mask].index.tolist()
    np.random.shuffle(users_idx)
    n_test = len(users_idx) // 2
    discount_df.loc[users_idx[:n_test], 'group'] = 'test'

# генерируем данные эксперимента
for _, row in discount_df.iterrows():
    visits = generate_experiment_visits(
        row['user_id'], row['discount_percent'], row['group'],
        EXPERIMENT_START, EXPERIMENT_END
    )
    all_experiment_records.extend(visits)

df_history = pd.DataFrame(all_history_records)
df_experiment = pd.DataFrame(all_experiment_records)

# добавляем пропуски
missing_mask = np.random.random(len(df_experiment)) < 0.003
df_experiment.loc[missing_mask, 'order_amount'] = np.nan

# сортировка
df_history = df_history.sort_values('datetime').reset_index(drop=True)
df_experiment = df_experiment.sort_values('datetime').reset_index(drop=True)

# сохранение
df_history.to_csv('historical_data.csv', index=False)
df_experiment.to_csv('experiment_data.csv', index=False)

print("\nDataset generation is completed")
# проверка распределеения пользователей по стратам
print("\nDistribution:")
print(discount_df.groupby(['discount_percent', 'group']).size())


Dataset generation is completed

Distribution:
discount_percent  group  
0                 control     405
                  test        404
3                 control     541
                  test        541
5                 control     546
                  test        545
7                 control    1509
                  test       1509
dtype: int64
